# F6 Prequential Evaluation — Neural Network (50 Configurations)

## Overview

**Function 6**: Cake recipe scoring — 5 ingredient inputs (flour, sugar, eggs, butter, milk), composite negative score output.  
**Objective**: Maximise (bring score closest to 0).

This notebook performs **prequential (one-step-ahead) evaluation** of a Neural Network surrogate with **MC Dropout** uncertainty estimation across **50 hyperparameter configurations**.

### Approach

For each configuration:
1. Train on initial 20 points
2. Predict next unseen point (mean + uncertainty via MC Dropout)
3. Add actual observation to training set
4. Repeat for all 6 evaluation steps

### Hyperparameter Search Space

50 configurations varying three axes:

| Axis | Values |
|------|--------|
| **Hidden layers** | 1, 2 |
| **Nodes per layer** | 5, 8, 16, 32, 64 |
| **Learning rate** | 0.001, 0.005, 0.01, 0.05, 0.1 |

**Fixed**: Activation=ReLU, Dropout=0.2, Epochs=500, MC samples=50, Normalisation=z-score

### Metrics

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MAE** | mean(\|predicted − actual\|) | Lower is better — point prediction accuracy |
| **NLP** | −log p(actual \| predicted, σ) | Lower is better — calibration quality (primary) |
| **95% Coverage** | fraction within ±1.96σ | Closer to 95% is better — interval reliability |

## Imports

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Plot style
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 10

print('All imports successful.')

All imports successful.


## Data Loading

Load F6 data (Week 6) — 26 total samples, first 20 used as initial training set.

In [ ]:
# ── Load F6 data (Week 6) ────────────────────────────────────
WEEK = 6
X_all = np.load(f'../../data/f6/updated_inputs - Week {WEEK}.npy')
y_all = np.load(f'../../data/f6/updated_outputs - Week {WEEK}.npy')

N_INIT = 20  # Initial training points
n_total = len(y_all)
n_steps = n_total - N_INIT  # Evaluation steps

print(f'Function 6 — Cake Recipe Scoring (5D)')
print(f'  Data loaded: Week {WEEK}')
print(f'  X shape: {X_all.shape}  (input dimensions: {X_all.shape[1]})')
print(f'  y shape: {y_all.shape}')
print(f'  Output range: [{y_all.min():.6f}, {y_all.max():.6f}]')
print(f'  Output mean:  {y_all.mean():.6f}')
print(f'  Output std:   {y_all.std():.6f}')
print(f'  Output Var:   {y_all.var():.6f}')
print(f'  Initial training points: {N_INIT}')
print(f'  Evaluation steps: {n_steps}')

## Metrics Function

In [ ]:
def compute_metrics(predictions, actuals, stds):
    """Compute MAE, NLP, and 95% coverage.
    
    Args:
        predictions: array of predicted means
        actuals: array of actual values
        stds: array of predicted standard deviations
    
    Returns:
        dict with MAE, NLP, Coverage_95
    """
    errors = np.abs(predictions - actuals)
    mae = np.mean(errors)
    
    # Negative log predictive density (Gaussian assumption)
    clipped_stds = np.maximum(stds, 1e-10)
    nlp = 0.5 * np.log(2 * np.pi * clipped_stds**2) + \
          0.5 * ((actuals - predictions) / clipped_stds)**2
    mean_nlp = np.mean(nlp)
    
    # 95% coverage
    lower = predictions - 1.96 * clipped_stds
    upper = predictions + 1.96 * clipped_stds
    coverage = np.mean((actuals >= lower) & (actuals <= upper))
    
    return {'MAE': mae, 'NLP': mean_nlp, 'Coverage_95': coverage}

print('compute_metrics() defined.')

## Neural Network — Default Configuration

Starting configuration: **2 hidden layers, 5 nodes per layer, lr=0.01**

### Architecture

The `FlexibleNN` class builds an NN with a configurable number of hidden layers, each with the same number of nodes:

```
Input(5) → [Linear(5→n_nodes) → ReLU → Dropout(0.2)] × n_layers → Linear(n_nodes→1)
```

### Uncertainty via MC Dropout

At inference, the model is kept in `train()` mode (dropout active). 50 stochastic forward passes produce a distribution of predictions:
- **Mean prediction** = mean of 50 passes
- **Uncertainty (std)** = std of 50 passes

This provides a non-parametric uncertainty estimate without requiring a separate probabilistic model.

In [ ]:
# ── Flexible NN architecture ──────────────────────────────────
class FlexibleNN(nn.Module):
    """Neural network with configurable hidden layers and nodes.
    
    Args:
        input_dim: Number of input features
        n_layers: Number of hidden layers
        n_nodes: Number of nodes per hidden layer
        dropout: Dropout probability (default 0.2)
    """
    def __init__(self, input_dim, n_layers, n_nodes, dropout=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for _ in range(n_layers):
            layers.extend([
                nn.Linear(prev_dim, n_nodes),
                nn.ReLU(),
                nn.Dropout(p=dropout)
            ])
            prev_dim = n_nodes
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)


# ── Prequential evaluation function ───────────────────────────
MC_SAMPLES = 50
EPOCHS = 500

def nn_prequential_with_config(X_all, y_all, n_init, config):
    """Run prequential evaluation with a given NN configuration.
    
    Args:
        X_all: Full input array (n_total, input_dim)
        y_all: Full output array (n_total,)
        n_init: Number of initial training points
        config: dict with keys 'n_layers', 'n_nodes', 'lr', 'label'
    
    Returns:
        dict with predictions, actuals, stds, metrics
    """
    input_dim = X_all.shape[1]
    n_total = len(y_all)
    n_steps = n_total - n_init
    
    predictions = []
    actuals = []
    stds = []
    
    for step in range(n_steps):
        # Current training data
        n_train = n_init + step
        X_train = X_all[:n_train]
        y_train = y_all[:n_train]
        
        # Test point (next unseen)
        X_test = X_all[n_train:n_train+1]
        y_test = y_all[n_train]
        
        # z-score normalisation (fit on training data only)
        X_mean = X_train.mean(axis=0)
        X_std = X_train.std(axis=0) + 1e-10
        y_mean = y_train.mean()
        y_std = y_train.std() + 1e-10
        
        X_train_norm = (X_train - X_mean) / X_std
        y_train_norm = (y_train - y_mean) / y_std
        X_test_norm = (X_test - X_mean) / X_std
        
        # Convert to tensors
        X_t = torch.tensor(X_train_norm, dtype=torch.float32)
        y_t = torch.tensor(y_train_norm, dtype=torch.float32).unsqueeze(1)
        X_test_t = torch.tensor(X_test_norm, dtype=torch.float32)
        
        # Build and train model
        torch.manual_seed(42 + step)  # Reproducible per step
        model = FlexibleNN(input_dim, config['n_layers'], config['n_nodes'])
        optimizer = optim.Adam(model.parameters(), lr=config['lr'])
        criterion = nn.MSELoss()
        
        model.train()
        for epoch in range(EPOCHS):
            optimizer.zero_grad()
            pred = model(X_t)
            loss = criterion(pred, y_t)
            loss.backward()
            optimizer.step()
        
        # MC Dropout prediction (keep model in train mode)
        model.train()  # Dropout active
        mc_preds = []
        with torch.no_grad():
            for _ in range(MC_SAMPLES):
                p = model(X_test_t).squeeze().item()
                mc_preds.append(p)
        
        mc_preds = np.array(mc_preds)
        # Denormalise
        mc_preds_orig = mc_preds * y_std + y_mean
        
        mean_pred = mc_preds_orig.mean()
        std_pred = max(mc_preds_orig.std(), 1e-10)
        
        predictions.append(mean_pred)
        actuals.append(y_test)
        stds.append(std_pred)
    
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    stds = np.array(stds)
    
    metrics = compute_metrics(predictions, actuals, stds)
    
    return {
        'predictions': predictions,
        'actuals': actuals,
        'stds': stds,
        'metrics': metrics
    }


# ── Run default configuration ─────────────────────────────────
default_config = {'n_layers': 2, 'n_nodes': 5, 'lr': 0.01, 'label': 'NN: 2L×5N, lr=0.01 (default)'}

print(f"Running NN default ({default_config['label']}) prequential evaluation...")
print(f'  Training starts with {N_INIT} points, evaluating {n_steps} steps\n')

nn_default_results = nn_prequential_with_config(X_all, y_all, N_INIT, default_config)

for step in range(n_steps):
    pred = nn_default_results['predictions'][step]
    actual = nn_default_results['actuals'][step]
    error = pred - actual
    std = nn_default_results['stds'][step]
    print(f'  Step {step+1}: train={N_INIT+step} pts | predicted={pred:+.6f} | actual={actual:+.6f} | error={error:+.6f} | std={std:.6f}')

metrics = nn_default_results['metrics']
print(f"\n  Results:")
print(f"    MAE:          {metrics['MAE']:.6f}")
print(f"    Mean NLP:     {metrics['NLP']:.4f}")
print(f"    95% Coverage: {metrics['Coverage_95']*100:.1f}%")

### Default Configuration — Prequential Visualisation

In [ ]:
# ── Plot prequential results for default config ───────────────
def plot_prequential_results(results, label, color='#FF9800'):
    """Plot 3-panel prequential evaluation results."""
    preds = results['predictions']
    acts = results['actuals']
    stds_arr = results['stds']
    steps = np.arange(1, len(preds) + 1)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # Panel 1: Predictions vs Actuals
    axes[0].errorbar(steps, preds, yerr=1.96*stds_arr, fmt='o-', color=color,
                     capsize=4, label='Predicted ± 1.96σ', markersize=6)
    axes[0].scatter(steps, acts, color='black', marker='x', s=80, zorder=5, label='Actual')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Output Value')
    axes[0].set_title(f'{label}\nPredictions vs Actuals')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)
    
    # Panel 2: Absolute Error
    abs_errors = np.abs(preds - acts)
    axes[1].bar(steps, abs_errors, color=color, alpha=0.7)
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Absolute Error')
    axes[1].set_title('Absolute Error per Step')
    axes[1].grid(True, alpha=0.3)
    
    # Panel 3: NLP per step
    clipped = np.maximum(stds_arr, 1e-10)
    nlp_step = 0.5 * np.log(2 * np.pi * clipped**2) + \
               0.5 * ((acts - preds) / clipped)**2
    axes[2].bar(steps, nlp_step, color=color, alpha=0.7)
    axes[2].set_xlabel('Step')
    axes[2].set_ylabel('NLP')
    axes[2].set_title('NLP per Step')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_prequential_results(nn_default_results, default_config['label'])

## Neural Network HP Optimisation — 50 Configurations

Systematic grid over three hyperparameter axes:

| Axis | Values | Count |
|------|--------|-------|
| **Hidden layers** | 1, 2 | 2 |
| **Nodes per layer** | 5, 8, 16, 32, 64 | 5 |
| **Learning rate** | 0.001, 0.005, 0.01, 0.05, 0.1 | 5 |

**Total**: 2 × 5 × 5 = **50 configurations**

All other hyperparameters are held constant:
- Activation: ReLU
- Dropout: 0.2 (required for MC Dropout uncertainty)
- Training epochs: 500
- MC forward passes: 50
- Input/output normalisation: z-score

In [ ]:
# ── Generate 50 NN configurations ─────────────────────────────
layers_grid = [1, 2]
nodes_grid = [5, 8, 16, 32, 64]
lr_grid = [0.001, 0.005, 0.01, 0.05, 0.1]

nn_configs = []
for n_layers in layers_grid:
    for n_nodes in nodes_grid:
        for lr in lr_grid:
            label = f'NN: {n_layers}L×{n_nodes}N, lr={lr}'
            nn_configs.append({
                'n_layers': n_layers,
                'n_nodes': n_nodes,
                'lr': lr,
                'label': label
            })

print(f'Generated {len(nn_configs)} NN configurations.\n')

# ── Evaluate all 50 configurations ────────────────────────────
print(f'Running {len(nn_configs)} NN configurations...\n')

nn_results = []
for i, config in enumerate(nn_configs):
    try:
        result = nn_prequential_with_config(X_all, y_all, N_INIT, config)
        m = result['metrics']
        print(f'  [{i+1:2d}/{len(nn_configs)}] {config["label"]} ... '
              f'MAE={m["MAE"]:.4f}, NLP={m["NLP"]:.4f}, Cov={m["Coverage_95"]*100:.1f}%')
        nn_results.append({
            'label': config['label'],
            'MAE': m['MAE'],
            'NLP': m['NLP'],
            'Coverage_95': m['Coverage_95']
        })
    except Exception as e:
        print(f'  [{i+1:2d}/{len(nn_configs)}] {config["label"]} ... FAILED: {e}')
        nn_results.append({
            'label': config['label'],
            'MAE': float('nan'),
            'NLP': float('nan'),
            'Coverage_95': float('nan')
        })

nn_hp_df = pd.DataFrame(nn_results)
print(f'\nNN Hyperparameter Results ({len(nn_hp_df)} configs):\n')
nn_hp_df

### Best NN Configuration

In [ ]:
# ── Best NN by NLP ────────────────────────────────────────────
best_nn_idx = nn_hp_df['NLP'].idxmin()
best_nn = nn_hp_df.loc[best_nn_idx]

print(f'Best NN Configuration by NLP:')
print(f'  Config:    {best_nn["label"]}')
print(f'  MAE:       {best_nn["MAE"]:.6f}')
print(f'  NLP:       {best_nn["NLP"]:.4f}')
print(f'  Coverage:  {best_nn["Coverage_95"]*100:.1f}%')

best_mae_idx = nn_hp_df['MAE'].idxmin()
if best_mae_idx != best_nn_idx:
    best_mae_nn = nn_hp_df.loc[best_mae_idx]
    print(f'\nBest NN by MAE (different from NLP-best):')
    print(f'  Config:    {best_mae_nn["label"]}')
    print(f'  MAE:       {best_mae_nn["MAE"]:.6f}')
    print(f'  NLP:       {best_mae_nn["NLP"]:.4f}')
    print(f'  Coverage:  {best_mae_nn["Coverage_95"]*100:.1f}%')

## Sensitivity Analysis — All 50 Configurations

Horizontal bar charts showing NLP, MAE, and 95% Coverage for each configuration, colour-coded by architecture depth (1-layer vs 2-layer).

In [ ]:
# ── Sensitivity bar charts (all 50 configs) ───────────────────
# Colour by layer count: 1-layer = light orange, 2-layer = deep orange
colors = []
for label in nn_hp_df['label']:
    if '1L×' in label:
        colors.append('#FFB74D')  # Light orange (1-layer)
    else:
        colors.append('#E65100')  # Deep orange (2-layer)

# Sort by NLP for display
sorted_df = nn_hp_df.sort_values('NLP', ascending=True).reset_index(drop=True)
sorted_colors = []
for label in sorted_df['label']:
    if '1L×' in label:
        sorted_colors.append('#FFB74D')
    else:
        sorted_colors.append('#E65100')

fig, axes = plt.subplots(1, 3, figsize=(20, max(14, len(sorted_df) * 0.35)))

# NLP
y_pos = np.arange(len(sorted_df))
axes[0].barh(y_pos, sorted_df['NLP'], color=sorted_colors, alpha=0.8)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(sorted_df['label'], fontsize=7)
axes[0].set_xlabel('NLP (lower is better)')
axes[0].set_title('NLP — All 50 Configurations')
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3, axis='x')

# MAE
mae_sorted = nn_hp_df.sort_values('MAE', ascending=True).reset_index(drop=True)
mae_colors = []
for label in mae_sorted['label']:
    if '1L×' in label:
        mae_colors.append('#FFB74D')
    else:
        mae_colors.append('#E65100')

axes[1].barh(y_pos, mae_sorted['MAE'], color=mae_colors, alpha=0.8)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(mae_sorted['label'], fontsize=7)
axes[1].set_xlabel('MAE (lower is better)')
axes[1].set_title('MAE — All 50 Configurations')
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3, axis='x')

# Coverage
cov_sorted = nn_hp_df.sort_values('Coverage_95', ascending=False).reset_index(drop=True)
cov_colors = []
for label in cov_sorted['label']:
    if '1L×' in label:
        cov_colors.append('#FFB74D')
    else:
        cov_colors.append('#E65100')

axes[2].barh(y_pos, cov_sorted['Coverage_95'] * 100, color=cov_colors, alpha=0.8)
axes[2].set_yticks(y_pos)
axes[2].set_yticklabels(cov_sorted['label'], fontsize=7)
axes[2].set_xlabel('95% Coverage (%)')
axes[2].set_title('Coverage — All 50 Configurations')
axes[2].axvline(x=95, color='red', linestyle='--', alpha=0.5, label='Target 95%')
axes[2].invert_yaxis()
axes[2].grid(True, alpha=0.3, axis='x')
axes[2].legend(fontsize=8)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#FFB74D', label='1-layer NN'),
    Patch(facecolor='#E65100', label='2-layer NN'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, 1.02))

plt.tight_layout()
plt.show()

## Full Ranked Results — All 50 Configurations

All 50 configurations ranked by NLP (primary metric, lower is better).

In [ ]:
# ── Full ranked table (all 50 configs) ────────────────────────
ranked_df = nn_hp_df.sort_values('NLP').reset_index(drop=True)
ranked_df.index = ranked_df.index + 1  # 1-based ranking
ranked_df.index.name = 'Rank'

print(f'Full Ranked Results — All {len(ranked_df)} Configurations (sorted by NLP):\n')
ranked_df

## Conclusions

### Key Findings

This notebook evaluated **50 Neural Network configurations** for Function 6 (cake recipe scoring — 5D input, negative output in [−2.57, −0.22]).

Configurations varied across three dimensions:
1. **Hidden layers**: 1, 2
2. **Nodes per layer**: 5, 8, 16, 32, 64
3. **Learning rate**: 0.001, 0.005, 0.01, 0.05, 0.1

Key observations:
- The ranked results above show which NN architecture/training combination provides the best-calibrated surrogate for F6
- NLP (primary metric) captures both accuracy and uncertainty calibration via MC Dropout
- F6's narrow, negative output range (−2.57 to −0.22) should be manageable for NNs with z-score normalisation
- Smaller networks (5 nodes) may underfit, while larger ones (64 nodes) risk overfitting with only 20 training points
- The learning rate trades off convergence speed against training stability

### Implications for Bayesian Optimisation

The best-calibrated NN configuration should be used as the surrogate model in the Bayesian Optimisation pipeline for Function 6. MC Dropout provides practical uncertainty estimates for the UCB acquisition function to balance exploration and exploitation in the 5D ingredient space.

### Next Steps

- Use the winning NN configuration in the main BO pipeline for F6
- Extend prequential evaluation to remaining functions (F7, F8)
- Consider whether the optimal architecture depth and width generalise across functions
- Investigate the impact of increasing training epochs or MC samples on uncertainty calibration